# BioHub — sonde légère des données

Ce notebook inspecte les métadonnées et quelques frames sous-échantillonnées. Il ne charge jamais les 87+ Go en mémoire et n'entraîne aucun modèle.

La première cellule installe `zarr` depuis des wheels attachées au notebook. À défaut, elle tente PyPI uniquement si l'accès Internet est activé.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import subprocess
import sys

def ensure_zarr() -> None:
    try:
        import zarr  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    input_root = Path("/kaggle/input")
    wheel_dirs = sorted({
        wheel.parent
        for wheel in input_root.rglob("*.whl")
        if wheel.name.lower().startswith(
            ("zarr-", "numcodecs-", "donfig-", "google_crc32c-", "crc32c-")
        )
    })

    errors = []

    if wheel_dirs:
        command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
        ]
        for directory in wheel_dirs:
            command.extend(["--find-links", str(directory)])
        command.extend(["zarr<3", "numcodecs<0.16"])
        try:
            subprocess.check_call(command)
            importlib.invalidate_caches()
            import zarr  # noqa: F401
            return
        except Exception as exc:
            errors.append(f"installation offline: {exc}")

    try:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "zarr<3",
                "numcodecs<0.16",
            ]
        )
        importlib.invalidate_caches()
        import zarr  # noqa: F401
        return
    except Exception as exc:
        errors.append(f"installation PyPI: {exc}")

    details = "\n".join(f"- {error}" for error in errors)
    raise RuntimeError(
        "Le package zarr est absent. Pour ce notebook d'exploration, active Internet "
        "dans Notebook options puis relance cette cellule, ou attache un Dataset Kaggle "
        "contenant les wheels zarr 2.x et numcodecs.\n"
        f"{details}"
    )

ensure_zarr()

import zarr
print("zarr version:", zarr.__version__)


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMPETITION_SLUG = "biohub-cell-tracking-during-development"
CANDIDATE_ROOTS = [
    Path("/kaggle/input/competitions") / COMPETITION_SLUG,
    Path("/kaggle/input") / COMPETITION_SLUG,
]

DATA_ROOT = next((p for p in CANDIDATE_ROOTS if p.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Competition mount not found. Attach the BioHub competition data to this notebook."
    )

TRAIN_ROOT = DATA_ROOT / "train"
TEST_ROOT = DATA_ROOT / "test"
WORK_ROOT = Path("/kaggle/working")
print("DATA_ROOT:", DATA_ROOT)


In [ ]:
def discover_stores(root: Path, suffix: str) -> list[Path]:
    if not root.exists():
        return []
    return sorted(path for path in root.iterdir() if path.name.endswith(suffix))

train_zarr = discover_stores(TRAIN_ROOT, ".zarr")
train_geff = discover_stores(TRAIN_ROOT, ".geff")
test_zarr = discover_stores(TEST_ROOT, ".zarr")

print(f"train zarr: {len(train_zarr)}")
print(f"train geff: {len(train_geff)}")
print(f"test zarr : {len(test_zarr)}")
print("train examples:", [p.name for p in train_zarr[:5]])
print("test examples :", [p.name for p in test_zarr[:5]])


In [ ]:
def iter_arrays(group, prefix: str = ""):
    for key in group.keys():
        item = group[key]
        path = f"{prefix}/{key}" if prefix else str(key)
        if hasattr(item, "shape") and hasattr(item, "dtype"):
            yield path, item
        elif hasattr(item, "keys"):
            yield from iter_arrays(item, path)

def open_primary_array(store_path: Path):
    root = zarr.open(str(store_path), mode="r")
    if hasattr(root, "shape"):
        return "", root
    arrays = list(iter_arrays(root))
    if not arrays:
        raise RuntimeError(f"No array found in {store_path}")
    arrays.sort(key=lambda pair: np.prod(pair[1].shape), reverse=True)
    return arrays[0]

rows = []
for split, stores in (("train", train_zarr), ("test", test_zarr)):
    for store in stores:
        array_path, array = open_primary_array(store)
        rows.append(
            {
                "split": split,
                "dataset": store.stem,
                "array_path": array_path,
                "shape": tuple(int(v) for v in array.shape),
                "chunks": tuple(int(v) for v in array.chunks) if array.chunks else None,
                "dtype": str(array.dtype),
                "nbytes_logical_gb": float(
                    np.prod(array.shape) * np.dtype(array.dtype).itemsize / 1e9
                ),
            }
        )

inventory = pd.DataFrame(rows)
display(inventory)
inventory.to_csv(WORK_ROOT / "biohub_data_inventory.csv", index=False)


In [ ]:
if not train_zarr:
    raise RuntimeError("No training Zarr stores found.")

SAMPLE_STORE = train_zarr[0]
ARRAY_PATH, ARRAY = open_primary_array(SAMPLE_STORE)
print("Sample:", SAMPLE_STORE.name, "array:", ARRAY_PATH, "shape:", ARRAY.shape)

if len(ARRAY.shape) != 4:
    raise ValueError(f"Expected (T, Z, Y, X), got {ARRAY.shape}")

T, Z, Y, X = map(int, ARRAY.shape)
sampled_times = sorted({0, T // 2, T - 1})
stride_z = max(1, Z // 64)
stride_y = max(1, Y // 512)
stride_x = max(1, X // 512)

sample_rows = []
projections = {}
for t in sampled_times:
    volume = np.asarray(
        ARRAY[t, ::stride_z, ::stride_y, ::stride_x],
        dtype=np.float32,
    )
    finite = volume[np.isfinite(volume)]
    quantiles = (
        np.quantile(finite, [0.5, 0.9, 0.99, 0.999])
        if finite.size
        else [np.nan] * 4
    )
    sample_rows.append(
        {
            "dataset": SAMPLE_STORE.stem,
            "t": t,
            "sample_shape": volume.shape,
            "q50": quantiles[0],
            "q90": quantiles[1],
            "q99": quantiles[2],
            "q999": quantiles[3],
            "mean": float(np.nanmean(volume)),
            "std": float(np.nanstd(volume)),
        }
    )
    projections[t] = np.nanmax(volume, axis=0)

sample_stats = pd.DataFrame(sample_rows)
display(sample_stats)
sample_stats.to_csv(WORK_ROOT / "biohub_sample_intensity_stats.csv", index=False)


In [ ]:
for t, projection in projections.items():
    plt.figure(figsize=(8, 8))
    lo, hi = np.nanquantile(projection, [0.01, 0.999])
    plt.imshow(projection, vmin=lo, vmax=hi)
    plt.title(f"{SAMPLE_STORE.stem} — t={t} — projection max Z sous-échantillonnée")
    plt.axis("off")
    plt.show()


In [ ]:
report = {
    "data_root": str(DATA_ROOT),
    "zarr_version": zarr.__version__,
    "train_zarr_count": len(train_zarr),
    "train_geff_count": len(train_geff),
    "test_zarr_count": len(test_zarr),
    "sample_dataset": SAMPLE_STORE.stem,
    "sample_array_path": ARRAY_PATH,
    "sample_shape": list(map(int, ARRAY.shape)),
    "sampled_times": sampled_times,
    "strides_zyx": [stride_z, stride_y, stride_x],
}
(WORK_ROOT / "biohub_data_probe.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("Artifacts written to /kaggle/working")
